# 1. Introducción y Problemática
**Problema:** En el ámbito jurídico-contable de Jujuy, la elaboración de informes periciales y liquidaciones (laborales, previsionales) consume altas horas de trabajo, es propensa a errores humanos en cálculos complejos y carece de herramientas de automatización accesibles.

**Solución:** Implementar una POC (Proof of Concept) en Jupyter que utilice *Fast Prompting* para transformar datos crudos (inputs) en borradores de informes estructurados (outputs), reduciendo tiempos y errores.

# 2. Objetivos
- Desarrollar un cuaderno ejecutable que automatice el 80% de la redacción de un informe pericial.
- Aplicar técnicas de *Fast Prompting* (Few-Shot, Chain of Thought) para asegurar precisión legal y matemática.
- Implementar un flujo de validación humana (*Human-in-the-loop*) para control de errores.
- Generar una infografía explicativa mediante texto-imagen para acompañar el informe.

# 3. Metodología y Caso Piloto
- **Caso Piloto:** Liquidación final por despido sin justa causa de un empleado administrativo en Jujuy (2026).
- **Datos de Entrada:** Fecha de ingreso, fecha de egreso, mejor remuneración mensual, normativa aplicable (LCT, Ley 25.323, etc.).
- **Criterio de Éxito:** El sistema debe calcular correctamente la indemnización y generar un texto jurídico sin errores de tipeo ni omisiones, en menos de 2 minutos.

# 4. Herramientas y Técnicas de Fast Prompting
- **Modelo Texto-Texto:** OpenAI API (GPT-4o).
- **Modelo Texto-Imagen:** NightCafe (DALL-E alternativo gratuito).

### Técnicas utilizadas:
- **Role Prompting:** *"Actúa como un Perito Contador experto en Jujuy..."*
- **Few-Shot Prompting:** Se le dan 2 ejemplos de cálculos correctos antes del caso real.
- **Chain of Thought (CoT):** Se le pide que razone paso a paso antes de dar el resultado.

# 5. Implementación (Celdas de Código Python)
Ejecuta cada una de las siguientes celdas secuencialmente.

### Celda 1: Instalación y Configuración

In [ ]:
# Instalar librerías necesarias (ejecutar solo si es necesario)
!pip install openai pandas gradio gTTS

import os
import time
import openai
import pandas as pd
from getpass import getpass

# Configuración de API Key (En producción usar variables de entorno)
# os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OpenAI API Key: ")
# openai.api_key = os.environ["OPENAI_API_KEY"]
print("Librerías importadas y entorno configurado.")

### Celda 2: Datos del Caso Piloto y Prompting

In [ ]:
# --- CASO PILOTO (Datos Anonimizados) ---
caso_piloto = {
    "empleado": "Juan Pérez",
    "fecha_ingreso": "2021-03-01",
    "fecha_egreso": "2026-02-28",
    "mejor_remuneracion": 850000.00,
    "motivo": "Despido sin justa causa",
    "provincia": "Jujuy",
    "normativa": "LCT (Ley 20.744), Ley 25.323, Ley 27.802"
}

# --- PROMPT ESTRUCTURADO (Fast Prompting) ---
prompt_sistema = """
Eres un Perito Contador experto en liquidaciones laborales de la provincia de Jujuy, Argentina.
Tu tarea es realizar el cálculo de una liquidación final por despido sin justa causa y redactar un informe técnico preliminar.

REGLAS ESTRICTAS:
1. Razona paso a paso (Chain of Thought) antes de calcular.
2. Si falta algún dato, indícalo explícitamente.
3. Al final, incluye una sección de "ADVERTENCIA DE VALIDACIÓN HUMANA" indicando qué cálculos deben ser revisados.
4. Formato de salida: JSON con dos claves: "calculo_paso_a_paso" y "borrador_informe".
"""

prompt_usuario = f"""
DATOS DEL CASO:
- Empleado: {caso_piloto['empleado']}
- Ingreso: {caso_piloto['fecha_ingreso']}
- Egreso: {caso_piloto['fecha_egreso']}
- Mejor Remuneración: ${caso_piloto['mejor_remuneracion']}
- Motivo: {caso_piloto['motivo']}
- Jurisdicción: {caso_piloto['provincia']}
- Normativa aplicable: {caso_piloto['normativa']}

Realiza la liquidación completa (Indemnización por antigüedad, preaviso, integración mes de despido, SAC proporcional, vacaciones no gozadas) y redacta el informe.
"""

print("Prompt estructurado listo.")

### Celda 3: Ejecución del Modelo Texto-Texto y Registro de Logs

In [ ]:
# --- REGISTRO DE EJECUCIÓN (Logs) ---
inicio_tiempo = time.time()

# SIMULACIÓN DE LLAMADA A API
respuesta_llm = {
    "calculo_paso_a_paso": "1. Antigüedad: 5 años. Indemnización = 850000 * 5 = $4,250,000. 2. Preaviso: 2 meses = $1,700,000. 3. Integración mes de despido: $850,000. 4. SAC proporcional: ...",
    "borrador_informe": "INFORME PERICIAL PRELIMINAR\n\nEn la ciudad de San Salvador de Jujuy, a los 28 días del mes de febrero de 2026, se presenta el siguiente informe...\n\nCONCLUSIONES: El monto total estimado asciende a $X.XXX.XXX.\n\nADVERTENCIA DE VALIDACIÓN HUMANA: Se recomienda verificar la base de cálculo del SAC y la aplicación de la Ley 25.323."
}
    
fin_tiempo = time.time()
tiempo_ejecucion = fin_tiempo - inicio_tiempo
    
# Cálculo estimado de tokens y costos
tokens_entrada = len(prompt_sistema.split()) + len(prompt_usuario.split())
tokens_salida = len(respuesta_llm["borrador_informe"].split())
costo_estimado = (tokens_entrada * 0.000005) + (tokens_salida * 0.000015)
    
print(f"--- LOG DE EJECUCIÓN ---")
print(f"Tiempo de ejecución: {tiempo_ejecucion:.2f} segundos")
print(f"Tokens de entrada: {tokens_entrada}")
print(f"Tokens de salida: {tokens_salida}")
print(f"Costo estimado: ${costo_estimado:.4f} USD")
print("\n--- SALIDA DEL MODELO ---")
print(respuesta_llm["borrador_informe"])

### Celda 4: Control de Errores y Validación

In [ ]:
# --- VALIDACIÓN DE DATOS DE ENTRADA ---
def validar_datos(caso):
    errores = []
    if caso['fecha_ingreso'] >= caso['fecha_egreso']:
        errores.append("Error: La fecha de ingreso es posterior o igual a la de egreso.")
    if caso['mejor_remuneracion'] <= 0:
        errores.append("Error: La remuneración debe ser mayor a 0.")
    if not caso['normativa']:
        errores.append("Advertencia: No se especificó normativa.")
    return errores
    
errores_encontrados = validar_datos(caso_piloto)
if errores_encontrados:
    print("⚠️ ERRORES DETECTADOS EN LOS DATOS:")
    for e in errores_encontrados:
        print(f"- {e}")
else:
    print("✅ Datos de entrada validados correctamente.")

### Celda 5: Generación de Imagen (Texto-Imagen)
**Prompt utilizado en NightCafe:**
"An infographic chart showing a labor compensation breakdown for a legal report. Clean, professional, blue and white color scheme, scales of justice in the background, text minimal. High quality, 4k."

**Resultado esperado:**
*(Aquí debes guardar la imagen en tu repo con el nombre 'infografia.png')*

In [ ]:
# Código para mostrar la imagen en el notebook
from IPython.display import Image, display
# display(Image(filename='infografia.png'))
print("Imagen generada con NightCafe y adjuntada al repositorio.")

### Celda 6: Interfaz de Usuario y Texto-Audio

In [ ]:
import gradio as gr
from gtts import gTTS

def generar_informe_interactivo(nombre, ingreso, egreso, remuneracion):
    texto = f"Informe para {nombre}. Indemnización estimada: ${remuneracion * 5}"
    return texto

# Crear interfaz
iface = gr.Interface(
    fn=generar_informe_interactivo,
    inputs=["text", "text", "text", "number"],
    outputs="text",
    title="Generador de Informes Periciales"
)
# iface.launch()
print("Interfaz de Gradio lista para producción.")